In [ ]:
from pathlib import Path

RUN = "runs/plan-a-cross-fold"

# Works when the kernel starts in the repository root or in notebooks/.
ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "stratscout").is_dir() and (p / "AGENTS.md").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the CV-DL repository or its notebooks folder.")

run_dir = (ROOT / RUN).resolve()
if not run_dir.is_dir():
    raise FileNotFoundError(
        f"Run folder does not exist: {run_dir}\n"
        "Change RUN to an existing cross-fold run, for example "
        "'runs/plan-a-cross-fold-20261108', or create the default run.\n"
        "From the repository root, with the .venv environment active, run:\n"
        "python -m stratscout tactic-baseline "
        "--manifest data/processed/datasets/2026-11-01/train/manifest.csv "
        f'--output "{RUN}" --exclude-uncertain'
    )
print(f"Reading saved results from {run_dir}")

# Plan A results

1. Select the project's **.venv** Python kernel. If it is missing, install `ipykernel` in that environment.
2. Set `RUN` in the first cell to an existing cross-fold run, then run the cells from top to bottom.
3. Keep the figures on screen; this notebook does not write images, change results or train a model.

This view uses the two validation matches and seeds 0, 1 and 2 from the default Plan A run.
It reads saved JSON files only. It never opens clips, checkpoints or the test split.
Cross-fold results still reuse both matches, so they are development scores rather than independent test results.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

plt.switch_backend("module://matplotlib_inline.backend_inline")
plt.rcParams.update(
    {
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "figure.dpi": 110,
    }
)

config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
if config.get("split") != "train" or config.get("epoch_mode") != "cross-fold":
    raise ValueError("Choose a completed cross-fold run made from the training manifest.")

folds = {}
for path in sorted(run_dir.glob("fold-*.json")):
    if path.stem.removeprefix("fold-").isdigit():
        saved = json.loads(path.read_text(encoding="utf-8"))
        folds[saved["fold"]] = saved
if set(folds) != {1, 2}:
    raise ValueError("This notebook expects the two complete Furia/Spirit validation folds.")

runs_by_fold = {number: {r["seed"]: r for r in fold["runs"]} for number, fold in folds.items()}
for number, runs in runs_by_fold.items():
    if not {0, 1, 2}.issubset(runs):
        raise ValueError(f"Fold {number} is missing seeds 0, 1 or 2; use a default three-seed run.")
    if any(
        r["epoch_selection"] != "cross-fold" or r.get("optimistic") is None for r in runs.values()
    ):
        raise ValueError("Both cross-fold and optimistic comparison scores are required.")

samples = json.loads((run_dir / "samples.json").read_text(encoding="utf-8"))
if any(sample["split"] != "train" for sample in samples):
    raise ValueError("The saved samples must all belong to the training split.")
match_folders = {sample["match_id"]: sample["match_folder"] for sample in samples}
names = {"furia": "Furia vs Falcons", "spirit": "Spirit vs Falcons"}
fold_names = {}
for number, fold in folds.items():
    folder = match_folders[fold["validation_match_id"]]
    fold_names[number] = names.get(folder.split("-")[0], folder.replace("-", " ").title())

classes = config["classes"]
class_names = [label.replace("_", " ").capitalize() for label in classes]
seeds = sorted(runs_by_fold[1])
if seeds != sorted(runs_by_fold[2]):
    raise ValueError("The two folds must have the same seeds for this comparison.")
print("Class order:", ", ".join(class_names))
print("Validation matches:", ", ".join(fold_names.values()))
print("Seeds:", seeds)
print("Excluded clips:", config["excluded_clip_ids"] or "none")

### Fold 1: confusion matrix for seed 0

Counts on the diagonal are correct predictions. Off-diagonal counts show which classes are confused. Rows are true classes; columns are predicted classes.

In [ ]:
fold = folds[1]
matrix = np.asarray(runs_by_fold[1][0]["metrics"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(8.4, 6.8), layout="constrained")
heatmap = ax.imshow(matrix, cmap="Blues", vmin=0)
ax.set_xticks(range(len(classes)), class_names, rotation=30, ha="right")
ax.set_yticks(range(len(classes)), class_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(f"{fold_names[1]}: cross-fold confusion\nSeed 0; {matrix.sum()} validation clips")
for row in range(len(classes)):
    for column in range(len(classes)):
        ax.text(
            column,
            row,
            str(int(matrix[row, column])),
            ha="center",
            va="center",
            fontsize=12,
            color="white" if matrix[row, column] > matrix.max() / 2 else "black",
        )
colourbar = fig.colorbar(heatmap, ax=ax, label="Number of predictions")
colourbar.locator = MaxNLocator(integer=True)
colourbar.update_ticks()
plt.show()
plt.close(fig)

### Fold 1: confusion matrix summed over seeds

Look for mistakes that repeat across seeds. Each clip is counted once per seed, so these are prediction counts, not extra clips. Rows are true classes; columns are predicted classes.

In [ ]:
fold = folds[1]
matrix = np.sum(
    [run["metrics"]["confusion_matrix"] for run in runs_by_fold[1].values()],
    axis=0,
)
fig, ax = plt.subplots(figsize=(8.4, 6.8), layout="constrained")
heatmap = ax.imshow(matrix, cmap="Blues", vmin=0)
ax.set_xticks(range(len(classes)), class_names, rotation=30, ha="right")
ax.set_yticks(range(len(classes)), class_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(
    f"{fold_names[1]}: cross-fold confusion\n"
    f"Sum over {len(runs_by_fold[1])} seeds; {matrix.sum()} predictions"
)
for row in range(len(classes)):
    for column in range(len(classes)):
        ax.text(
            column,
            row,
            str(int(matrix[row, column])),
            ha="center",
            va="center",
            fontsize=12,
            color="white" if matrix[row, column] > matrix.max() / 2 else "black",
        )
colourbar = fig.colorbar(heatmap, ax=ax, label="Number of predictions")
colourbar.locator = MaxNLocator(integer=True)
colourbar.update_ticks()
plt.show()
plt.close(fig)

### Fold 2: confusion matrix for seed 0

Counts on the diagonal are correct predictions. Off-diagonal counts show which classes are confused. Rows are true classes; columns are predicted classes.

In [ ]:
fold = folds[2]
matrix = np.asarray(runs_by_fold[2][0]["metrics"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(8.4, 6.8), layout="constrained")
heatmap = ax.imshow(matrix, cmap="Blues", vmin=0)
ax.set_xticks(range(len(classes)), class_names, rotation=30, ha="right")
ax.set_yticks(range(len(classes)), class_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(f"{fold_names[2]}: cross-fold confusion\nSeed 0; {matrix.sum()} validation clips")
for row in range(len(classes)):
    for column in range(len(classes)):
        ax.text(
            column,
            row,
            str(int(matrix[row, column])),
            ha="center",
            va="center",
            fontsize=12,
            color="white" if matrix[row, column] > matrix.max() / 2 else "black",
        )
colourbar = fig.colorbar(heatmap, ax=ax, label="Number of predictions")
colourbar.locator = MaxNLocator(integer=True)
colourbar.update_ticks()
plt.show()
plt.close(fig)

### Fold 2: confusion matrix summed over seeds

Look for mistakes that repeat across seeds. Each clip is counted once per seed, so these are prediction counts, not extra clips. Rows are true classes; columns are predicted classes.

In [ ]:
fold = folds[2]
matrix = np.sum(
    [run["metrics"]["confusion_matrix"] for run in runs_by_fold[2].values()],
    axis=0,
)
fig, ax = plt.subplots(figsize=(8.4, 6.8), layout="constrained")
heatmap = ax.imshow(matrix, cmap="Blues", vmin=0)
ax.set_xticks(range(len(classes)), class_names, rotation=30, ha="right")
ax.set_yticks(range(len(classes)), class_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title(
    f"{fold_names[2]}: cross-fold confusion\n"
    f"Sum over {len(runs_by_fold[2])} seeds; {matrix.sum()} predictions"
)
for row in range(len(classes)):
    for column in range(len(classes)):
        ax.text(
            column,
            row,
            str(int(matrix[row, column])),
            ha="center",
            va="center",
            fontsize=12,
            color="white" if matrix[row, column] > matrix.max() / 2 else "black",
        )
colourbar = fig.colorbar(heatmap, ax=ax, label="Number of predictions")
colourbar.locator = MaxNLocator(integer=True)
colourbar.update_ticks()
plt.show()
plt.close(fig)

### Fold 1: F1 for each class

Compare cross-fold scores with the optimistic scores selected on the same validation match. Bars average the saved seed results. The count under each class is the number of validation clips; small counts make the score less reliable.

In [ ]:
fold = folds[1]
cross_f1 = np.mean(
    [
        [run["metrics"]["per_class"][label]["f1"] for label in classes]
        for run in runs_by_fold[1].values()
    ],
    axis=0,
)
optimistic_f1 = np.mean(
    [
        [run["optimistic"]["metrics"]["per_class"][label]["f1"] for label in classes]
        for run in runs_by_fold[1].values()
    ],
    axis=0,
)
positions = np.arange(len(classes))
labels = [
    f"{name}\n(n={fold['validation_counts'][label]})"
    for label, name in zip(classes, class_names, strict=True)
]
fig, ax = plt.subplots(figsize=(10, 5.5), layout="constrained")
cross_bars = ax.bar(positions - 0.19, cross_f1, 0.38, color="#0072B2", label="Cross-fold")
optimistic_bars = ax.bar(
    positions + 0.19,
    optimistic_f1,
    0.38,
    color="#D55E00",
    label="Optimistic (selected on validation)",
)
ax.bar_label(cross_bars, fmt="%.2f", padding=3, fontsize=9)
ax.bar_label(optimistic_bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_xticks(positions, labels)
ax.set_ylim(0, 1.1)
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_xlabel("Class (number of validation clips)")
ax.set_ylabel("Mean F1")
ax.set_title(f"{fold_names[1]}: per-class F1, mean over {len(runs_by_fold[1])} seeds")
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2)
ax.legend(loc="upper left")
plt.show()
plt.close(fig)

### Fold 2: F1 for each class

Compare cross-fold scores with the optimistic scores selected on the same validation match. Bars average the saved seed results. The count under each class is the number of validation clips; small counts make the score less reliable.

In [ ]:
fold = folds[2]
cross_f1 = np.mean(
    [
        [run["metrics"]["per_class"][label]["f1"] for label in classes]
        for run in runs_by_fold[2].values()
    ],
    axis=0,
)
optimistic_f1 = np.mean(
    [
        [run["optimistic"]["metrics"]["per_class"][label]["f1"] for label in classes]
        for run in runs_by_fold[2].values()
    ],
    axis=0,
)
positions = np.arange(len(classes))
labels = [
    f"{name}\n(n={fold['validation_counts'][label]})"
    for label, name in zip(classes, class_names, strict=True)
]
fig, ax = plt.subplots(figsize=(10, 5.5), layout="constrained")
cross_bars = ax.bar(positions - 0.19, cross_f1, 0.38, color="#0072B2", label="Cross-fold")
optimistic_bars = ax.bar(
    positions + 0.19,
    optimistic_f1,
    0.38,
    color="#D55E00",
    label="Optimistic (selected on validation)",
)
ax.bar_label(cross_bars, fmt="%.2f", padding=3, fontsize=9)
ax.bar_label(optimistic_bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_xticks(positions, labels)
ax.set_ylim(0, 1.1)
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_xlabel("Class (number of validation clips)")
ax.set_ylabel("Mean F1")
ax.set_title(f"{fold_names[2]}: per-class F1, mean over {len(runs_by_fold[2])} seeds")
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2)
ax.legend(loc="upper left")
plt.show()
plt.close(fig)

## Training curves

Cross-fold fitting records training loss at every epoch, but evaluates validation **only at the scored epoch**.
The dashed validation curves below come from the separate epoch search (`*-optimistic-history.json`).
They are not extra cross-fold measurements. The final cross-fold score is shown as a diamond, with a vertical line at its scored epoch.
Training loss uses class weights; validation loss is unweighted, so their values are not directly comparable.

### Fold 1, seed 0: loss and validation F1

Look for training loss falling while validation stops improving. The line marks the epoch borrowed from the other fold; the diamonds are the cross-fold measurements at that epoch. Dashed curves show the separate epoch search.

In [ ]:
run = runs_by_fold[1][0]
history = json.loads((run_dir / "fold-1-seed-0-history.json").read_text(encoding="utf-8"))
search_history = json.loads(
    (run_dir / "fold-1-seed-0-optimistic-history.json").read_text(encoding="utf-8")
)
scored_epoch = run["evaluation_epoch"]
scored = next(row for row in history if row["epoch"] == scored_epoch)
epochs = [row["epoch"] for row in history]
search_epochs = [row["epoch"] for row in search_history]
fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
loss_ax.plot(
    epochs,
    [row["train_loss"] for row in history],
    color="#0072B2",
    label="Cross-fold training loss",
)
loss_ax.plot(
    search_epochs,
    [row["validation_loss"] for row in search_history],
    color="#D55E00",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation loss",
)
loss_ax.scatter(
    scored_epoch,
    scored["validation_loss"],
    color="#D55E00",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
f1_ax.plot(
    search_epochs,
    [row["validation_macro_f1"] for row in search_history],
    color="#009E73",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation macro-F1",
)
f1_ax.scatter(
    scored_epoch,
    scored["validation_macro_f1"],
    color="#009E73",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
for ax in (loss_ax, f1_ax):
    ax.axvline(
        scored_epoch,
        color="#333333",
        linestyle="-.",
        linewidth=1.4,
        label=f"Scored epoch {scored_epoch}",
    )
    ax.set_xlabel("Epoch")
    ax.set_xlim(0, max(scored_epoch, max(search_epochs)) + 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
loss_ax.set_title("Training and validation loss")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_ylim(bottom=0)
f1_ax.set_title("Validation macro-F1")
f1_ax.set_ylabel("Macro-F1")
f1_ax.set_ylim(0, 1)
fig.suptitle(f"{fold_names[1]}: seed 0, saved training history", fontsize=15)
plt.show()
plt.close(fig)

### Fold 1, seed 1: loss and validation F1

Look for training loss falling while validation stops improving. The line marks the epoch borrowed from the other fold; the diamonds are the cross-fold measurements at that epoch. Dashed curves show the separate epoch search.

In [ ]:
run = runs_by_fold[1][1]
history = json.loads((run_dir / "fold-1-seed-1-history.json").read_text(encoding="utf-8"))
search_history = json.loads(
    (run_dir / "fold-1-seed-1-optimistic-history.json").read_text(encoding="utf-8")
)
scored_epoch = run["evaluation_epoch"]
scored = next(row for row in history if row["epoch"] == scored_epoch)
epochs = [row["epoch"] for row in history]
search_epochs = [row["epoch"] for row in search_history]
fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
loss_ax.plot(
    epochs,
    [row["train_loss"] for row in history],
    color="#0072B2",
    label="Cross-fold training loss",
)
loss_ax.plot(
    search_epochs,
    [row["validation_loss"] for row in search_history],
    color="#D55E00",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation loss",
)
loss_ax.scatter(
    scored_epoch,
    scored["validation_loss"],
    color="#D55E00",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
f1_ax.plot(
    search_epochs,
    [row["validation_macro_f1"] for row in search_history],
    color="#009E73",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation macro-F1",
)
f1_ax.scatter(
    scored_epoch,
    scored["validation_macro_f1"],
    color="#009E73",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
for ax in (loss_ax, f1_ax):
    ax.axvline(
        scored_epoch,
        color="#333333",
        linestyle="-.",
        linewidth=1.4,
        label=f"Scored epoch {scored_epoch}",
    )
    ax.set_xlabel("Epoch")
    ax.set_xlim(0, max(scored_epoch, max(search_epochs)) + 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
loss_ax.set_title("Training and validation loss")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_ylim(bottom=0)
f1_ax.set_title("Validation macro-F1")
f1_ax.set_ylabel("Macro-F1")
f1_ax.set_ylim(0, 1)
fig.suptitle(f"{fold_names[1]}: seed 1, saved training history", fontsize=15)
plt.show()
plt.close(fig)

### Fold 1, seed 2: loss and validation F1

Look for training loss falling while validation stops improving. The line marks the epoch borrowed from the other fold; the diamonds are the cross-fold measurements at that epoch. Dashed curves show the separate epoch search.

In [ ]:
run = runs_by_fold[1][2]
history = json.loads((run_dir / "fold-1-seed-2-history.json").read_text(encoding="utf-8"))
search_history = json.loads(
    (run_dir / "fold-1-seed-2-optimistic-history.json").read_text(encoding="utf-8")
)
scored_epoch = run["evaluation_epoch"]
scored = next(row for row in history if row["epoch"] == scored_epoch)
epochs = [row["epoch"] for row in history]
search_epochs = [row["epoch"] for row in search_history]
fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
loss_ax.plot(
    epochs,
    [row["train_loss"] for row in history],
    color="#0072B2",
    label="Cross-fold training loss",
)
loss_ax.plot(
    search_epochs,
    [row["validation_loss"] for row in search_history],
    color="#D55E00",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation loss",
)
loss_ax.scatter(
    scored_epoch,
    scored["validation_loss"],
    color="#D55E00",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
f1_ax.plot(
    search_epochs,
    [row["validation_macro_f1"] for row in search_history],
    color="#009E73",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation macro-F1",
)
f1_ax.scatter(
    scored_epoch,
    scored["validation_macro_f1"],
    color="#009E73",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
for ax in (loss_ax, f1_ax):
    ax.axvline(
        scored_epoch,
        color="#333333",
        linestyle="-.",
        linewidth=1.4,
        label=f"Scored epoch {scored_epoch}",
    )
    ax.set_xlabel("Epoch")
    ax.set_xlim(0, max(scored_epoch, max(search_epochs)) + 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
loss_ax.set_title("Training and validation loss")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_ylim(bottom=0)
f1_ax.set_title("Validation macro-F1")
f1_ax.set_ylabel("Macro-F1")
f1_ax.set_ylim(0, 1)
fig.suptitle(f"{fold_names[1]}: seed 2, saved training history", fontsize=15)
plt.show()
plt.close(fig)

### Fold 2, seed 0: loss and validation F1

Look for training loss falling while validation stops improving. The line marks the epoch borrowed from the other fold; the diamonds are the cross-fold measurements at that epoch. Dashed curves show the separate epoch search.

In [ ]:
run = runs_by_fold[2][0]
history = json.loads((run_dir / "fold-2-seed-0-history.json").read_text(encoding="utf-8"))
search_history = json.loads(
    (run_dir / "fold-2-seed-0-optimistic-history.json").read_text(encoding="utf-8")
)
scored_epoch = run["evaluation_epoch"]
scored = next(row for row in history if row["epoch"] == scored_epoch)
epochs = [row["epoch"] for row in history]
search_epochs = [row["epoch"] for row in search_history]
fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
loss_ax.plot(
    epochs,
    [row["train_loss"] for row in history],
    color="#0072B2",
    label="Cross-fold training loss",
)
loss_ax.plot(
    search_epochs,
    [row["validation_loss"] for row in search_history],
    color="#D55E00",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation loss",
)
loss_ax.scatter(
    scored_epoch,
    scored["validation_loss"],
    color="#D55E00",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
f1_ax.plot(
    search_epochs,
    [row["validation_macro_f1"] for row in search_history],
    color="#009E73",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation macro-F1",
)
f1_ax.scatter(
    scored_epoch,
    scored["validation_macro_f1"],
    color="#009E73",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
for ax in (loss_ax, f1_ax):
    ax.axvline(
        scored_epoch,
        color="#333333",
        linestyle="-.",
        linewidth=1.4,
        label=f"Scored epoch {scored_epoch}",
    )
    ax.set_xlabel("Epoch")
    ax.set_xlim(0, max(scored_epoch, max(search_epochs)) + 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
loss_ax.set_title("Training and validation loss")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_ylim(bottom=0)
f1_ax.set_title("Validation macro-F1")
f1_ax.set_ylabel("Macro-F1")
f1_ax.set_ylim(0, 1)
fig.suptitle(f"{fold_names[2]}: seed 0, saved training history", fontsize=15)
plt.show()
plt.close(fig)

### Fold 2, seed 1: loss and validation F1

Look for training loss falling while validation stops improving. The line marks the epoch borrowed from the other fold; the diamonds are the cross-fold measurements at that epoch. Dashed curves show the separate epoch search.

In [ ]:
run = runs_by_fold[2][1]
history = json.loads((run_dir / "fold-2-seed-1-history.json").read_text(encoding="utf-8"))
search_history = json.loads(
    (run_dir / "fold-2-seed-1-optimistic-history.json").read_text(encoding="utf-8")
)
scored_epoch = run["evaluation_epoch"]
scored = next(row for row in history if row["epoch"] == scored_epoch)
epochs = [row["epoch"] for row in history]
search_epochs = [row["epoch"] for row in search_history]
fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
loss_ax.plot(
    epochs,
    [row["train_loss"] for row in history],
    color="#0072B2",
    label="Cross-fold training loss",
)
loss_ax.plot(
    search_epochs,
    [row["validation_loss"] for row in search_history],
    color="#D55E00",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation loss",
)
loss_ax.scatter(
    scored_epoch,
    scored["validation_loss"],
    color="#D55E00",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
f1_ax.plot(
    search_epochs,
    [row["validation_macro_f1"] for row in search_history],
    color="#009E73",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation macro-F1",
)
f1_ax.scatter(
    scored_epoch,
    scored["validation_macro_f1"],
    color="#009E73",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
for ax in (loss_ax, f1_ax):
    ax.axvline(
        scored_epoch,
        color="#333333",
        linestyle="-.",
        linewidth=1.4,
        label=f"Scored epoch {scored_epoch}",
    )
    ax.set_xlabel("Epoch")
    ax.set_xlim(0, max(scored_epoch, max(search_epochs)) + 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
loss_ax.set_title("Training and validation loss")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_ylim(bottom=0)
f1_ax.set_title("Validation macro-F1")
f1_ax.set_ylabel("Macro-F1")
f1_ax.set_ylim(0, 1)
fig.suptitle(f"{fold_names[2]}: seed 1, saved training history", fontsize=15)
plt.show()
plt.close(fig)

### Fold 2, seed 2: loss and validation F1

Look for training loss falling while validation stops improving. The line marks the epoch borrowed from the other fold; the diamonds are the cross-fold measurements at that epoch. Dashed curves show the separate epoch search.

In [ ]:
run = runs_by_fold[2][2]
history = json.loads((run_dir / "fold-2-seed-2-history.json").read_text(encoding="utf-8"))
search_history = json.loads(
    (run_dir / "fold-2-seed-2-optimistic-history.json").read_text(encoding="utf-8")
)
scored_epoch = run["evaluation_epoch"]
scored = next(row for row in history if row["epoch"] == scored_epoch)
epochs = [row["epoch"] for row in history]
search_epochs = [row["epoch"] for row in search_history]
fig, (loss_ax, f1_ax) = plt.subplots(1, 2, figsize=(13, 4.9), layout="constrained")
loss_ax.plot(
    epochs,
    [row["train_loss"] for row in history],
    color="#0072B2",
    label="Cross-fold training loss",
)
loss_ax.plot(
    search_epochs,
    [row["validation_loss"] for row in search_history],
    color="#D55E00",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation loss",
)
loss_ax.scatter(
    scored_epoch,
    scored["validation_loss"],
    color="#D55E00",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
f1_ax.plot(
    search_epochs,
    [row["validation_macro_f1"] for row in search_history],
    color="#009E73",
    linestyle="--",
    alpha=0.85,
    label="Epoch-search validation macro-F1",
)
f1_ax.scatter(
    scored_epoch,
    scored["validation_macro_f1"],
    color="#009E73",
    edgecolor="black",
    marker="D",
    s=65,
    zorder=5,
    label="Cross-fold validation at scored epoch",
)
for ax in (loss_ax, f1_ax):
    ax.axvline(
        scored_epoch,
        color="#333333",
        linestyle="-.",
        linewidth=1.4,
        label=f"Scored epoch {scored_epoch}",
    )
    ax.set_xlabel("Epoch")
    ax.set_xlim(0, max(scored_epoch, max(search_epochs)) + 1)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
loss_ax.set_title("Training and validation loss")
loss_ax.set_ylabel("Cross-entropy loss")
loss_ax.set_ylim(bottom=0)
f1_ax.set_title("Validation macro-F1")
f1_ax.set_ylabel("Macro-F1")
f1_ax.set_ylim(0, 1)
fig.suptitle(f"{fold_names[2]}: seed 2, saved training history", fontsize=15)
plt.show()
plt.close(fig)

### Fold 1: macro-F1 across seeds

Look at how much the cross-fold score changes with initialisation. The dashed line is the majority-class guess chosen from the training match. Seed variation does not measure uncertainty on unseen matches.

In [ ]:
fold = folds[1]
scores = [runs_by_fold[1][seed]["metrics"]["macro_f1"] for seed in seeds]
majority = fold["majority"]["macro_f1"]
majority_class = fold["majority_class"].replace("_", " ").capitalize()
fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
ax.plot(seeds, scores, marker="o", markersize=8, color="#0072B2", label="Cross-fold macro-F1")
ax.axhline(
    majority,
    linestyle="--",
    color="#555555",
    label=f"Majority: {majority_class} (F1={majority:.3f})",
)
for seed, score in zip(seeds, scores, strict=True):
    ax.annotate(
        f"{score:.3f}", (seed, score), xytext=(0, 10), textcoords="offset points", ha="center"
    )
ax.set_xticks(seeds)
ax.set_ylim(0, 1)
ax.set_xlabel("Random seed")
ax.set_ylabel("Validation macro-F1")
ax.set_title(f"{fold_names[1]}: cross-fold macro-F1 by seed")
ax.grid(axis="y", alpha=0.2)
ax.legend(loc="upper right")
plt.show()
plt.close(fig)

### Fold 2: macro-F1 across seeds

Look at how much the cross-fold score changes with initialisation. The dashed line is the majority-class guess chosen from the training match. Seed variation does not measure uncertainty on unseen matches.

In [ ]:
fold = folds[2]
scores = [runs_by_fold[2][seed]["metrics"]["macro_f1"] for seed in seeds]
majority = fold["majority"]["macro_f1"]
majority_class = fold["majority_class"].replace("_", " ").capitalize()
fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
ax.plot(seeds, scores, marker="o", markersize=8, color="#0072B2", label="Cross-fold macro-F1")
ax.axhline(
    majority,
    linestyle="--",
    color="#555555",
    label=f"Majority: {majority_class} (F1={majority:.3f})",
)
for seed, score in zip(seeds, scores, strict=True):
    ax.annotate(
        f"{score:.3f}", (seed, score), xytext=(0, 10), textcoords="offset points", ha="center"
    )
ax.set_xticks(seeds)
ax.set_ylim(0, 1)
ax.set_xlabel("Random seed")
ax.set_ylabel("Validation macro-F1")
ax.set_title(f"{fold_names[2]}: cross-fold macro-F1 by seed")
ax.grid(axis="y", alpha=0.2)
ax.legend(loc="upper right")
plt.show()
plt.close(fig)

## Keep in mind

These are scores on hand-cut clips from two development matches. Cross-fold epoch selection still reuses both matches: each validation match trained the other fold's model. Neither these plots nor the seed spread establish performance on an unseen match. NaVi remains held out.

Small classes have very few clips. The sum of confusion matrices counts repeated predictions of the same clips, and the per-class bars are means of seed-level F1 scores, not F1 calculated from a pooled matrix.